# Vanguard Challenge
## Multi-Asset Portfolio Construction

In [4]:
%pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 71.8 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


In [5]:
import numpy as np
import pennylane as qp
from pennylane import numpy as pnp

### Parameters

| Parameter | Variable | Type | Constraints | Description |
| :--- | :---: | :---: | :--- | :--- |
| **Expected return** | $\boldsymbol{\mu}$ | $\mathbb{R}^N$ | $\mu_\min \le \mu_i \le \mu_\max$ | Annualized mean expected return for each asset |
| **Covariance matrix** | $\boldsymbol{\Sigma}$ | $\mathbb{R}^{N \times N}$ | Symmetric, Positive Definite | Asset risk, variances, and asset-to-asset co-movements |
| **Volatility cap** | $\sigma_{\text{max}}$ | $\mathbb{R}^+$ |  | Maximum allowable portfolio volatility threshold |
| **Transaction costs** | $\boldsymbol{c}$ | $\mathbb{R}^N$ | $c_i \ge 0$ | Friction cost per unit change during rebalancing $\vert{}w_i - w_{0,i}\vert{}$ |
| **Current holdings** | $\boldsymbol{w}_0$ | $\mathbb{R}^N$ | $\sum w_{0,i} = 1$ | Existing portfolio weights before optimization |
| **Risk aversion** | $q$ | $\mathbb{R}^+$ |  | Multiplier balancing return vs. covariance risk |
| **Liquidity penalty** | $\boldsymbol{\ell}$ | $\mathbb{R}^N$ | $\ell_i \propto \frac{1}{\text{Avg Daily Volume}_i}$ | Penalty score based on inverse average daily volume |
| **Sector allocation** | $S_m$ | Set | $S_m \subset \{1 \dots N\}$ | Partition mapping assets to market sectors (Tech, Energy, etc.) |
| **Sector exposure limit** | $L_m$ | $\mathbb{R}^+$ | $L_{k,\min}\le L_k\le L_{k,\max}$ | Maximum total allocation allowed in Sector $m$ |
| **Diversification cardinality** | $K$ | $\mathbb{Z}^+$ | $1 \le K \le N$ | Target number of assets to hold in the portfolio |

In [14]:
def generate_synthetic_dataset(N=10, M=3, K=None, volatility=None, new_client=False, seed=42):
    """
    Generates realistic market dataset parameters.
    @params: N (# assets), M (# sectors), K (target # assets)
    """
    np.random.seed(seed)

    # Expected returns: mu_i in [mu_min, mu_max]
    mu_min, mu_max = 0.03, 0.15
    mu = np.random.uniform(mu_min, mu_max, size=N)

    # Covariance Matrix using factor analyis: Sigma = B * B^T + D
    sig_min, sig_max = -0.2, 0.5
    var_min, var_max = 0.01, 0.08
    num_factors = max(2, M)
    B = np.random.uniform(sig_min, sig_max, size=(N, num_factors))  # loading
    D = np.diag(np.random.uniform(var_min, var_max, size=N))        # idiosyncratic risk
    Sigma = B @ B.T + D

    # Existing portfolio holdings (w0)
    if new_client:
        w0 = np.zeros(N)
    else:
        w0 = np.random.dirichlet(np.ones(N))
        w0 = np.where(w0 < 1/N, 0.0, w0)  # zero small entries
        w0 = w0 / np.sum(w0)

    # Transaction costs (c_i) per asset
    c_min, cmax = 0.001, 0.005
    c_trans = np.random.uniform(c_min, cmax, size=N)

    # Liquidity penalties (ell_i): higher for less liquid assets
    scale = 0.02
    avg_daily_volume = np.random.lognormal(mean=12, sigma=1.0, size=N)
    ell = 1.0 / (np.log10(avg_daily_volume) + 1e-5)
    ell = (ell - ell.min()) / (ell.max() - ell.min()) * scale

    # Sector allocations (S_i)
    sector_map = {}
    assets_per_sector = N // M
    for i in range(M):
        ini = i * assets_per_sector
        fin = (i + 1) * assets_per_sector if i < M - 1 else N
        sector_map[f"Sector_{i+1}"] = list(range(ini, fin))

    # Exposure Caps (L_i): Cap each sector to at most 40% of selected assets/weight
    sector_caps = {f"Sector_{i+1}": max(1, int(0.4 * N)) for i in range(M)}

    # Other parameters
    q_risk_aversion = 0.5
    if not volatility: volatility = 0.12    # 12% max volatility
    if not K: K = max(2, N // 3)            # Target 1/3 of assets
    w_min, w_max = 0.5 / ds['K'], min(1, 1.5 / ds['K'])     # Investment guardrails

    return {
        "N": N,
        "mu": mu,
        "Sigma": Sigma,
        "w0": w0,
        "c_trans": c_trans,
        "ell": ell,
        "S": sector_map,
        "L": sector_caps,
        "q": q_risk_aversion,
        "sigma_max": volatility,
        "K": K,
        "w_min": w_min,
        "w_max": w_max,
    }

In [15]:
N = 10      # Assets: Stock, Bond, Real Estate, Gold, ...
M = 3       # Number of sectors
ds = generate_synthetic_dataset(N, M)
ds

{'N': 10,
 'mu': array([0.07494481, 0.14408572, 0.11783927, 0.10183902, 0.04872224,
        0.04871934, 0.03697003, 0.13394114, 0.1021338 , 0.11496871]),
 'Sigma': array([[ 0.42883402, -0.05270554,  0.116908  ,  0.06945142,  0.07183993,
         -0.0325126 , -0.03384174,  0.11864783,  0.09196509,  0.19940848],
        [-0.05270554,  0.05771804, -0.02016558, -0.00947032, -0.01287686,
         -0.02503373, -0.01498108, -0.01787143, -0.0520007 , -0.02126833],
        [ 0.116908  , -0.02016558,  0.05105245,  0.02777375,  0.02171119,
          0.01083065, -0.00218055,  0.02062611,  0.06874918,  0.05603755],
        [ 0.06945142, -0.00947032,  0.02777375,  0.13626327,  0.00069985,
         -0.02877328, -0.06046599, -0.08308982,  0.08401211,  0.05211312],
        [ 0.07183993, -0.01287686,  0.02171119,  0.00069985,  0.04554216,
          0.01724921,  0.01837589,  0.04627481,  0.02437498,  0.02804751],
        [-0.0325126 , -0.02503373,  0.01083065, -0.02877328,  0.01724921,
          0.207828

### Mathematical formulation (Modern portfolio theory)

Using binary allocation variables $x_i \in \{0, 1\}$ for asset selection,

**Objective** - To balance return against covariance risk:
$$\min_{\boldsymbol{x}} \quad q \cdot \boldsymbol{x}^T \boldsymbol{\Sigma} \boldsymbol{x} - \boldsymbol{\mu}^T \boldsymbol{x}$$

**Constraints:**
1. **Transaction cost:** $\quad C_{\text{trans}} = \sum_{i=1}^N c_i (x_i - x_{0,i})^2\quad$ *(Rebalances distance from current holdings $x_0$)*
2. **Liquidity limit:** $\quad x_i \cdot w_i \le \ell_i \cdot V_i \quad \forall i\quad$ *(Penalizes illiquid assets based on daily trading volume $V_i$)*
3. **Diversification / Cardinality:** $\quad \sum_{i=1}^N x_i = K\quad $ *(Enforces holding exactly $K$ asset classes)*
4. **Sector/exposure limits:** $\quad \sum_{i \in S_m} x_i \le L_m \quad \forall m\quad$ *(Prevents over-exposure to Sector $m$)*
5. **Investment guardrails:** $\quad w_{\text{min}} \cdot x_i \le w_i \le w_{\text{max}} \cdot x_i\quad$ *(Enforces minimum position sizes and prevents dust allocations)*

---

### QUBO Formulation

Cost Hamiltonian $H(\boldsymbol{x}) = \boldsymbol{x}^T Q \boldsymbol{x}$:

$$H(\boldsymbol{x}) = \underbrace{q \cdot \boldsymbol{x}^T \boldsymbol{\Sigma} \boldsymbol{x}}_{\text{Portfolio variance}} - \underbrace{\boldsymbol{\mu}^T \boldsymbol{x}}_{\text{Expected return}} + \underbrace{\sum_{i=1}^N c_i (x_i - x_{0,i})^2}_{\text{Transaction costs}} + \underbrace{\boldsymbol{\ell}^T \boldsymbol{x}}_{\text{Liquidity penalty}}$$
$$\quad + \underbrace{P_1 \left(\sum_{i=1}^N x_i - K\right)^2}_{\text{Diversification penalty}} + \underbrace{P_2 \sum_{k} \max\left(0, \sum_{i \in S_k} x_i - L_k\right)^2}_{\text{Sector exposure penalty}}$$

In [55]:
def build_qubo(
    mu, Sigma, q, K, sector_map, sector_caps, c_trans, liquidity,
    w0, w_min, w_max, P_div=12, P_sec=8, P_guard=20
):
    """
    Constructs an MPT QUBO matrix Q incorporating:
    - Expected return (mu) & Volatility Risk (Sigma with risk aversion q)
    - Transaction costs (c_trans)
    - liquidity penalties (liquidity)
    - Diversification (K)
    - Sector exposure limits (sector_map & sector_caps)
    - Investment guardrails (w_min, w_max)
    """
    N = len(mu)
    Q = np.zeros((N, N))

    # Risk covariance
    for i in range(N):
        for j in range(i+1, N):
            Q[i, j] += 2 * q * Sigma[i, j]

    # Expected return
    for i in range(N):
        Q[i, i] -= mu[i]

    # Transaction cost
    for i in range(N):
        if w0[i] == 0:
            Q[i, i] += c_trans[i]
        else:
            Q[i, i] -= c_trans[i]

    # Liquidity penalty
    for i in range(N):
        Q[i, i] += liquidity[i]

    # Diversification penalty
    for i in range(N):
        Q[i, i] += P_div * (1 - 2 * K)
        for j in range(i + 1, N):
            Q[i, j] += 2 * P_div

    # Sector limit penalties
    for sector, assets in sector_map.items():
        limit = sector_caps[sector]
        for idx_a, i in enumerate(assets):
            if limit > 1:
                Q[i, i] += P_sec * (1.0 - 2.0 * limit)
            for j in assets[idx_a + 1:]:
                Q[i, j] += 2.0 * P_sec

    # Investment guardrails
    if w_min > 1 / K:
        penalty = P_guard * (w_min - 1 / K)**2
        for i in range(N):
            Q[i, i] += penalty

    if w_max < 1 / K:
        penalty = P_guard * (1 / K - w_max)**2
        for i in range(N):
            Q[i, i] += penalty

    return Q

In [56]:
Q = build_qubo(
    ds['mu'], ds['Sigma'], ds['q'], ds['K'], ds['S'], ds['L'],
    ds['c_trans'], ds['ell'], ds['w0'], ds['w_min'], ds['w_max']
)
with np.printoptions(precision=1, suppress=True, linewidth=120):
    print(f'{Q=}')

Q=array([[-116.1,   39.9,   40.1,   24.1,   24.1,   24. ,   24. ,   24.1,   24.1,   24.2],
       [   0. , -116.1,   40. ,   24. ,   24. ,   24. ,   24. ,   24. ,   23.9,   24. ],
       [   0. ,    0. , -116.1,   24. ,   24. ,   24. ,   24. ,   24. ,   24.1,   24.1],
       [   0. ,    0. ,    0. , -116.1,   40. ,   40. ,   23.9,   23.9,   24.1,   24.1],
       [   0. ,    0. ,    0. ,    0. , -116. ,   40. ,   24. ,   24. ,   24. ,   24. ],
       [   0. ,    0. ,    0. ,    0. ,    0. , -116. ,   24.1,   24.1,   24.1,   24. ],
       [   0. ,    0. ,    0. ,    0. ,    0. ,    0. , -116. ,   40.1,   40. ,   39.9],
       [   0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. , -116.1,   39.9,   40. ],
       [   0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. , -116.1,   40. ],
       [   0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. ,    0. , -116.1]])


### Ising Mapping

To run on QAOA or Quantum Annealers, convert binary variables $x_i \in \{0, 1\}$ to Pauli spin operators $s_i \in \{-1, +1\}$ using the substitution:

$$x_i = \frac{1 - Z_i}{2}$$

Substituting $x_i$ into $H(\boldsymbol{x})$ converts the QUBO matrix $Q$ into the Ising Spin Glass Hamiltonian:

$$H_{\text{Ising}} = \sum_{i=1}^N h_i Z_i + \sum_{i < j} J_{ij} Z_i Z_j + C_{\text{offset}}$$

where $h_i$ represents single-qubit longitudinal magnetic fields (biases), $J_{ij}$ represents two-qubit coupling strengths, and $Z_i$ is the Pauli-$Z$ operator.

In [57]:
def qubo_to_ising(Q):
    N = Q.shape[0]
    h = np.zeros(N)
    J = np.zeros((N, N))
    offset = 0.0

    for i in range(N):
        h[i] += 0.5 * Q[i, i]
        offset += 0.5 * Q[i, i]
        for j in range(i + 1, N):
            J[i, j] = 0.25 * Q[i, j]
            h[i] += 0.25 * Q[i, j]
            h[j] += 0.25 * Q[i, j]
            offset += 0.25 * Q[i, j]

    return h, J, offset

In [58]:
h, J, C = qubo_to_ising(Q)
with np.printoptions(precision=1, suppress=True, linewidth=120):
    print(f'{h=}\n{J=}\n{C=}')

h=array([4.1, 3.9, 4. , 4. , 4. , 4. , 8. , 8. , 8. , 8. ])
J=array([[ 0., 10., 10.,  6.,  6.,  6.,  6.,  6.,  6.,  6.],
       [ 0.,  0., 10.,  6.,  6.,  6.,  6.,  6.,  6.,  6.],
       [ 0.,  0.,  0.,  6.,  6.,  6.,  6.,  6.,  6.,  6.],
       [ 0.,  0.,  0.,  0., 10., 10.,  6.,  6.,  6.,  6.],
       [ 0.,  0.,  0.,  0.,  0., 10.,  6.,  6.,  6.,  6.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  6.,  6.,  6.,  6.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0., 10., 10., 10.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 10., 10.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 10.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]])
C=np.float64(-262.15849931247317)


### QAOA

In [59]:
# Define observables
obs_list = []
coeffs = []

# Add linear h_i * Z_i
for i in range(N):
    if abs(h[i]) > 1e-6:
        coeffs.append(h[i])
        obs_list.append(qp.PauliZ(i))

# Add interaction J_ij * Z_i Z_j
for i in range(N):
    for j in range(i + 1, N):
        if abs(J[i, j]) > 1e-6:
            coeffs.append(J[i, j])
            obs_list.append(qp.PauliZ(i) @ qp.PauliZ(j))

# Cost Hamiltonian
H_C = qp.Hamiltonian(coeffs, obs_list)

# Driver/Mixer Hamiltonian H_M = sum X_i
H_M = qp.Hamiltonian([1.0] * N, [qp.PauliX(i) for i in range(N)])

In [60]:
dev = qp.device("default.qubit", wires=N)

# QAOA layer
def qaoa_layer(gamma, beta):
    # Cost Unitary: U_C(gamma) = e^(-i * gamma * H_C)
    for i in range(N):
        if abs(h[i]) > 1e-6:
            qp.RZ(2.0 * gamma * h[i], wires=i)

    for i in range(N):
        for j in range(i + 1, N):
            if abs(J[i, j]) > 1e-6:
                qp.IsingZZ(2.0 * gamma * J[i, j], wires=[i, j])

    # Mixer Unitary U_M(beta) = e^(-i * beta * H_M)
    for i in range(N):
        qp.RX(2.0 * beta, wires=i)

# QAOA circuit QNode
@qp.qnode(dev)
def qaoa_circuit(params):
    # p = number of QAOA layers
    p = params.shape[1]

    # Mixer initialized in qual superposition |+>^N
    for i in range(N):
        qp.Hadamard(wires=i)

    # Apply p alternating QAOA layers
    for l in range(p):
        qaoa_layer(params[0][l], params[1][l])

    # Measure expectation value of cost Hamiltonian
    return qp.expval(H_C)

In [61]:
# Depth p = 4 layers
p = 4

# Initial parameters: params[0] -> gammas, params[1] -> betas
np.random.seed(42)
init_gammas = np.random.uniform(0.01, 0.05, size=p)
init_betas = np.random.uniform(0.01, np.pi / 4, size=p)
init_params = pnp.array([init_gammas, init_betas], requires_grad=True)

optimizer = qp.AdamOptimizer(stepsize=0.005)
params = init_params

print("--- Executing QAOA Optimization ---")
for step in range(1, 81):
    params, cost = optimizer.step_and_cost(qaoa_circuit, params)
    total_energy = cost + C

    if step % 20 == 0 or step == 1:
        print(f"Step {step:02d} | Cost: {cost:.4f} | Total QUBO Energy: {total_energy:.4f}")

--- Executing QAOA Optimization ---
Step 01 | Cost: 53.2877 | Total QUBO Energy: -208.8708
Step 20 | Cost: 19.9912 | Total QUBO Energy: -242.1673
Step 40 | Cost: 8.2183 | Total QUBO Energy: -253.9402
Step 60 | Cost: 5.6463 | Total QUBO Energy: -256.5122
Step 80 | Cost: 5.2052 | Total QUBO Energy: -256.9533


In [54]:
@qp.qnode(dev)
def probability_circuit(params):
    p = params.shape[1]
    for i in range(N):
        qp.Hadamard(wires=i)
    for l in range(p):
        qaoa_layer(params[0][l], params[1][l])
    return qp.probs(wires=range(N))

probs = probability_circuit(params)

top_indices = np.argsort(probs)[::-1][:5]

print("Top 5 sampled portfolios")
for idx in top_indices:
    bitstr = format(idx, f'0{N}b')
    prob = probs[idx] * 100

    # Map bitstring (0/1) to Ising spin z (+1/-1)
    x = np.array([int(b) for b in bitstr])
    z = 1 - 2 * x

    # Exact energy evaluation
    energy = np.dot(h, z) + np.dot(z, np.dot(J, z)) + C
    k_selected = np.sum(x)

    print(f"Bitstring: {bitstr} | Prob: {prob:5.1f}% | Energy: {energy:8.2f} | K: {k_selected}")

Top 5 sampled portfolios
Bitstring: 1000100110 | Prob:   0.4% | Energy: -1800.54 | K: 4
Bitstring: 1000101001 | Prob:   0.4% | Energy: -1800.53 | K: 4
Bitstring: 1000010110 | Prob:   0.4% | Energy: -1800.51 | K: 4
Bitstring: 1000100101 | Prob:   0.4% | Energy: -1800.23 | K: 4
Bitstring: 1000011001 | Prob:   0.4% | Energy: -1800.60 | K: 4


In [62]:
# Brute-force exact energy evaluation for all 1024 states
min_energy = float('inf')
best_bitstr = None
best_k = None

for idx in range(1024):
    bitstr = format(idx, f'010b')
    x = np.array([int(b) for b in bitstr])

    # Map x in {0,1} to Z in {+1, -1}
    z = 1 - 2 * x

    # Energy: Z^T J Z + h^T Z + C
    energy = np.dot(h, z) + np.dot(z, np.dot(J, z)) + C

    if energy < min_energy:
        min_energy = energy
        best_bitstr = bitstr
        best_k = np.sum(x)

print(f"True Classical Ground State of (h, J, C):")
print(f"Bitstring: {best_bitstr} | Energy: {min_energy:.4f} | K: {best_k}")

True Classical Ground State of (h, J, C):
Bitstring: 1000111001 | Energy: -308.6523 | K: 5
